# EDSS 취업 코호트 학교유형·시도 층화 분석


## tl;dr

2016→2020 원천 보고 취업자 비율은 학교유형 4개와 시도 계열 18개 층 모두에서 하락했다. 같은 층에 계속 남은 학교만 사용한 균형 패널에서도 22/22 방향이 일치했다. 이 비율은 공식 취업률이 아니며 인과효과를 의미하지 않는다.


## Context & Methods

### Key Assumptions

- 2010–2013년 6월 1일과 2014–2020년 12월 31일 구간을 분리한다. 2013→2014는 비교하지 않는다.
- 비율은 학교별 평균이 아니라 층별 취업자 합계 ÷ 졸업자 합계다.
- 단일 시도는 원천 `provinces`를 사용하고 복수 캠퍼스는 `복수시도`, 결측은 `속성없음`으로 따로 보존한다.
- 층 고정 균형 표본은 구간 모든 코호트에서 같은 비결측 층에 속한 학교만 포함한다.


In [1]:
import csv
import hashlib
import importlib.util
import json
import os
from pathlib import Path

import duckdb
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = Path(os.environ.get('EDSS_DUCKDB_PATH', REPO_ROOT / 'data/processed/edss/restricted/edss_all.duckdb'))
SCRIPT = REPO_ROOT / 'scripts/analyze_edss_employment_stratified_trends.py'
QUALITY_CSV = REPO_ROOT / 'data/metadata/edss_employment_stratified_attribute_quality.csv'
STABILITY_CSV = REPO_ROOT / 'data/metadata/edss_employment_stratified_attribute_stability.csv'
TRENDS_CSV = REPO_ROOT / 'data/metadata/edss_employment_stratified_trends.csv'
JSON_PATH = REPO_ROOT / 'data/metadata/edss_employment_stratified_trends.json'

spec = importlib.util.spec_from_file_location('stratified', SCRIPT)
stratified = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(stratified)
assert DB.is_file(), DB


## Data

제한 DuckDB의 학교–취업 코호트 결합 뷰를 읽기 전용으로 다시 조회한다. OpenID는 품질·균형 자격 검사에만 사용하고 출력하지 않는다.


In [2]:
connection = duckdb.connect(str(DB), read_only=True)
try:
    source_rows = stratified.query_source_rows(connection)
finally:
    connection.close()

source_profile = stratified.validate_source_rows(source_rows)
quality = stratified.build_quality_rows(source_rows)
stability = stratified.build_stability_rows(source_rows)
trends = stratified.build_stratified_rows(source_rows)
validation = stratified.validate_outputs(source_rows, trends)
display(source_profile, validation)


{'row_count': 5969,
 'blank_open_id_row_count': 0,
 'duplicate_school_cohort_key_count': 0,
 'negative_measure_value_count': 0,
 'cohort_count': 11,
 'first_cohort_year': '2010',
 'last_cohort_year': '2020'}

{'unique_attribute_stratum_cohort_keys': True,
 'school_type_strata_reconcile_to_source_by_cohort': True,
 'province_strata_reconcile_to_source_by_cohort': True,
 'interval_start_changes_are_null': True}

## Results

학교유형과 시도의 결측·안정성을 전체 학교 수와 졸업자 가중 포괄률로 검사한다.


In [3]:
missing_2012 = [
    row for row in quality
    if row['employment_cohort_year'] == '2012'
]
assert {row['missing_attribute_school_count'] for row in missing_2012} == {3}
assert {row['matched_attribute_reported_graduate_count'] for row in missing_2012} == {564933}

lines = [
    '| 속성 | 비교 구간 | 균형 학교 | 안정 학교 | 안정률 | 졸업자 포괄률 최솟값 |',
    '|---|---|---:|---:|---:|---:|',
]
for row in stability:
    label = '학교유형' if row['attribute'] == 'school_type' else '시도'
    lines.append(
        f"| {label} | {row['comparison_interval']} | "
        f"{row['interval_balanced_school_count']:,} | "
        f"{row['stable_attribute_balanced_school_count']:,} | "
        f"{row['stable_attribute_school_share']:.2%} | "
        f"{row['minimum_stable_attribute_graduate_coverage_share']:.2%} |"
    )
display(Markdown('\n'.join(lines)))


| 속성 | 비교 구간 | 균형 학교 | 안정 학교 | 안정률 | 졸업자 포괄률 최솟값 |
|---|---|---:|---:|---:|---:|
| 시도 | december_31_2014_2020 | 515 | 505 | 98.06% | 98.13% |
| 시도 | june_1_2010_2013 | 496 | 484 | 97.58% | 96.91% |
| 학교유형 | december_31_2014_2020 | 515 | 515 | 100.00% | 100.00% |
| 학교유형 | june_1_2010_2013 | 496 | 496 | 100.00% | 100.00% |

In [4]:
summary = json.loads(JSON_PATH.read_text(encoding='utf-8'))
changes = summary['findings']['changes_2016_to_2020_by_attribute']
lines = [
    '| 학교유형 | 전체 2016→2020 | 층 고정 균형 | 균형 학교 |',
    '|---|---:|---:|---:|',
]
for row in changes['school_type']:
    lines.append(
        f"| {row['stratum']} | {row['all_available_change_pp']:+.4f}%p | "
        f"{row['stratum_balanced_change_pp']:+.4f}%p | "
        f"{row['balanced_school_count']:,} |"
    )
display(Markdown('\n'.join(lines)))
assert summary['findings']['all_vs_balanced_direction_agreement_count'] == 22
assert summary['findings']['all_vs_balanced_comparable_stratum_count'] == 22


| 학교유형 | 전체 2016→2020 | 층 고정 균형 | 균형 학교 |
|---|---:|---:|---:|
| 대학 | -3.4008%p | -3.3504%p | 190 |
| 대학원 | -2.0243%p | -2.0115%p | 164 |
| 대학원대학 | -4.0541%p | -4.0541%p | 1 |
| 전문대학 | -4.4276%p | -4.4582%p | 160 |

## Takeaways

학교유형은 구간 균형 학교 내에서 100% 안정적이었고, 시도는 97.58%·98.06%가 구간 내내 같았다. 2016→2020 하락은 학교유형·시도에 광범위하게 퍼져 있었고 층 이동이나 학교 진입·이탈만으로 방향을 설명하기 어렵다. 소표본 층은 크기·순위 해석에 적합하지 않고, 균형 패널은 생존편향과 지표 정의 변화를 해소하지 않는다.


In [5]:
with QUALITY_CSV.open(encoding='utf-8', newline='') as handle:
    committed_quality = list(csv.DictReader(handle))
with STABILITY_CSV.open(encoding='utf-8', newline='') as handle:
    committed_stability = list(csv.DictReader(handle))
with TRENDS_CSV.open(encoding='utf-8', newline='') as handle:
    committed_trends = list(csv.DictReader(handle))

assert len(committed_quality) == len(quality) == 22
assert len(committed_stability) == len(stability) == 4
assert len(committed_trends) == len(trends) == 241
for key, path in (
    ('attribute_quality_csv', QUALITY_CSV),
    ('attribute_stability_csv', STABILITY_CSV),
    ('stratified_trends_csv', TRENDS_CSV),
):
    expected = summary['outputs'][key]['sha256']
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == expected

display(Markdown('✅ 원천 재집계, 합계 재조정, 비교 구간, 3개 CSV 행 수·SHA-256 검산을 통과했습니다.'))


✅ 원천 재집계, 합계 재조정, 비교 구간, 3개 CSV 행 수·SHA-256 검산을 통과했습니다.